# Trabajo 15: Descarga y Preparación de Datasets (GAVD y UR Fall)

Este cuaderno automatiza la descarga, verificación y estructuración de los dos conjuntos de datos en vídeo RGB necesarios para el sistema de detección de episodios de bloqueo de la marcha (*Freezing of Gait*):

1. **GAVD (*Gait Abnormality in Video Dataset*):** Descarga las anotaciones clínicas abiertas desde GitHub y recorta los clips de vídeo correspondientes a marcha parkinsoniana y marcha normal.
2. **UR Fall Detection Dataset:** Descarga secuencias grabadas con cámara fija en interiores domésticos (actividades cotidianas como caminar, detenerse o sentarse, junto con caídas) y las compila en formato `.mp4`.

El cuaderno comprueba previamente si los metadatos y los vídeos ya existen en disco para evitar descargas redundantes.

### Estructura de directorios
```text
data/
├── raw/
│   ├── gavd/
│   │   ├── annotations/      # CSVs originales con bounding boxes y etiquetas
│   │   └── clips/            # Vídeos recortados (.mp4)
│   └── ur_fall/
│       └── clips/            # Secuencias compiladas (.mp4)
└── metadata/
    ├── metadata_gavd.csv     # Índice unificado de GAVD
    └── metadata_urfall.csv   # Índice unificado de UR Fall
```

In [1]:
%pip install -q requests pandas yt-dlp opencv-python tqdm urllib3 kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuración e Inputs

Define en esta sección todas las rutas, límites de descarga y parámetros de control. Si `FORCE_DOWNLOAD` es `False`, el cuaderno omitirá la descarga de cualquier dataset cuyos metadatos y vídeos ya estén completos en disco.

In [2]:
import json
import os
import re
import shutil
import zipfile
from getpass import getpass
from pathlib import Path

import cv2
import pandas as pd
import requests
import urllib3
import yt_dlp
from tqdm.auto import tqdm

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- PARÁMETROS DE CONTROL ---
FORCE_DOWNLOAD = False  # Cambiar a True para forzar la re-descarga completa

# --- RUTAS DEL PROYECTO ---
BASE_DIR = Path('data')
META_DIR = BASE_DIR / 'metadata'

GAVD_ANN_DIR = BASE_DIR / 'raw' / 'gavd' / 'annotations'
GAVD_CLIPS_DIR = BASE_DIR / 'raw' / 'gavd' / 'clips'
META_GAVD_PATH = META_DIR / 'metadata_gavd.csv'

URFALL_CLIPS_DIR = BASE_DIR / 'raw' / 'ur_fall' / 'clips'
META_URFALL_PATH = META_DIR / 'metadata_urfall.csv'

# --- PARÁMETROS DATASET 1: GAVD ---
GAVD_CLASES = ['parkinsons', 'normal']
GAVD_MAX_CLIPS_POR_CLASE = 25  # Usar None para descargar todas las secuencias disponibles

# --- PARÁMETROS DATASET 2: UR FALL ---
URFALL_MAX_ADL = 25   # Actividades diarias normales (caminar, detenerse, sentarse)
URFALL_MAX_FALL = 10  # Secuencias de caídas y pérdida de equilibrio
URFALL_FPS = 30.0

HTTP_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

for carpeta in [META_DIR, GAVD_ANN_DIR, GAVD_CLIPS_DIR, URFALL_CLIPS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print(f"Directorio base configurado: {BASE_DIR.resolve()}")

Directorio base configurado: D:\universidad\cuarto\primer_cuatri\entornos_inteligentes\trabajo_final\data


d:\universidad\cuarto\primer_cuatri\entornos_inteligentes\trabajo_final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Funciones Auxiliares de Verificación

Funciones compartidas para comprobar si un dataset ya está descargado íntegramente y para ordenar los fotogramas de forma natural antes de compilar vídeo.

In [3]:
def dataset_ya_descargado(meta_path: Path, col_ruta: str = 'clip_path') -> bool:
    """Verifica que el CSV de metadatos exista, tenga filas y todos sus vídeos estén en disco."""
    if not meta_path.exists():
        return False
    try:
        df = pd.read_csv(meta_path)
    except Exception:
        return False
    if df.empty or col_ruta not in df.columns:
        return False
    return all(Path(p).exists() and Path(p).stat().st_size > 1024 for p in df[col_ruta])


def orden_natural(path_obj: Path) -> list:
    """Ordena nombres de archivo respetando índices numéricos (ej. frame-2 antes que frame-10)."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', path_obj.name)]


def compilar_frames_a_mp4(carpeta_frames: Path, salida_mp4: Path, fps: float = 30.0) -> bool:
    """Convierte una secuencia de imágenes .png/.jpg en un archivo de vídeo .mp4."""
    frames = sorted(
        [p for p in carpeta_frames.rglob('*') if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg')],
        key=orden_natural
    )
    if len(frames) < 10:
        return False

    first_img = cv2.imread(str(frames[0]))
    if first_img is None:
        return False

    h, w = first_img.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(salida_mp4), fourcc, fps, (w, h))
    for f_path in frames:
        img = cv2.imread(str(f_path))
        if img is not None:
            out.write(img)
    out.release()
    return salida_mp4.exists() and salida_mp4.stat().st_size > 1024

## 3. Dataset 1: GAVD (*Gait Abnormality in Video Dataset*)

Localiza los archivos CSV de anotaciones en el repositorio público de GitHub, descarga los vídeos fuente mediante `yt-dlp` y recorta únicamente los intervalos de fotogramas anotados para las clases seleccionadas.

In [4]:
def obtener_anotaciones_gavd(destino: Path) -> list[Path]:
    """Carga los CSV locales si existen o los descarga desde el repositorio de GitHub."""
    locales = sorted(destino.glob('*.csv'))
    if locales and not FORCE_DOWNLOAD:
        return locales

    api_url = 'https://api.github.com/search/repositories?q=GAVD+Gait+Abnormality+in:name,description'
    resp = requests.get(api_url, timeout=15)
    resp.raise_for_status()
    items = resp.json().get('items', [])

    csv_urls = []
    for repo in items[:3]:
        full_name = repo['full_name']
        branch = repo.get('default_branch', 'main')
        tree_url = f'https://api.github.com/repos/{full_name}/git/trees/{branch}?recursive=1'
        tree_resp = requests.get(tree_url, timeout=15)
        if tree_resp.status_code == 200:
            for node in tree_resp.json().get('tree', []):
                if node['path'].endswith('.csv') and ('GAVD' in node['path'] or 'annotation' in node['path'].lower()):
                    raw_url = f"https://raw.githubusercontent.com/{full_name}/{branch}/{node['path']}"
                    csv_urls.append((Path(node['path']).name, raw_url))
        if csv_urls:
            break

    if not csv_urls:
        raise RuntimeError(
            'No se encontraron archivos CSV de GAVD en GitHub. '
            f'Colócalos manualmente en {destino}'
        )

    archivos = []
    for nombre, url in csv_urls:
        ruta_salida = destino / nombre
        if not ruta_salida.exists() or FORCE_DOWNLOAD:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            ruta_salida.write_bytes(r.content)
        archivos.append(ruta_salida)
    return archivos


def descargar_y_recortar_gavd(df: pd.DataFrame, clips_dir: Path, clases: list[str], max_clips: int | None) -> pd.DataFrame:
    """Descarga los vídeos fuente de GAVD y recorta cada secuencia según sus fotogramas inicial y final."""
    temp_dir = clips_dir / '_temp_full_videos'
    temp_dir.mkdir(exist_ok=True)

    df_filtrado = df[df['gait_pat'].str.lower().isin([c.lower() for c in clases])].copy()
    secuencias = []
    for seq_id, grupo in df_filtrado.groupby('seq'):
        secuencias.append({
            'seq_id': seq_id,
            'gait_pat': grupo['gait_pat'].iloc[0].lower(),
            'url': grupo['url'].iloc[0],
            'frame_start': int(grupo['frame_num'].min()),
            'frame_end': int(grupo['frame_num'].max())
        })

    df_seqs = pd.DataFrame(secuencias)
    if max_clips is not None:
        df_seqs = df_seqs.groupby('gait_pat').head(max_clips).reset_index(drop=True)

    ydl_opts = {
        'format': 'bestvideo[height<=720][ext=mp4]/best[height<=720][ext=mp4]/best',
        'outtmpl': str(temp_dir / '%(id)s.%(ext)s'),
        'quiet': True,
        'no_warnings': True
    }

    registros = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for _, row in tqdm(df_seqs.iterrows(), total=len(df_seqs), desc='Procesando clips GAVD'):
            clip_path = clips_dir / f"{row['gait_pat']}_{row['seq_id']}.mp4"
            if clip_path.exists() and not FORCE_DOWNLOAD:
                registros.append({**row.to_dict(), 'clip_path': str(clip_path)})
                continue

            try:
                info = ydl.extract_info(row['url'], download=True)
                video_completo = Path(ydl.prepare_filename(info))

                cap = cv2.VideoCapture(str(video_completo))
                fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

                out = cv2.VideoWriter(str(clip_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
                cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_start'])
                for _ in range(row['frame_start'], row['frame_end'] + 1):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    out.write(frame)
                cap.release()
                out.release()

                if clip_path.exists() and clip_path.stat().st_size > 1024:
                    registros.append({**row.to_dict(), 'clip_path': str(clip_path)})
            except Exception:
                continue

    shutil.rmtree(temp_dir, ignore_errors=True)
    return pd.DataFrame(registros)

In [5]:
if dataset_ya_descargado(META_GAVD_PATH) and not FORCE_DOWNLOAD:
    meta_gavd = pd.read_csv(META_GAVD_PATH)
    print(f"[OMITIDO] GAVD ya está descargado ({len(meta_gavd)} secuencias en {META_GAVD_PATH}).")
else:
    print("Descargando anotaciones y clips de GAVD...")
    csv_gavd = obtener_anotaciones_gavd(GAVD_ANN_DIR)
    df_gavd_raw = pd.concat([pd.read_csv(f) for f in csv_gavd], ignore_index=True)
    meta_gavd = descargar_y_recortar_gavd(
        df=df_gavd_raw,
        clips_dir=GAVD_CLIPS_DIR,
        clases=GAVD_CLASES,
        max_clips=GAVD_MAX_CLIPS_POR_CLASE
    )
    meta_gavd.to_csv(META_GAVD_PATH, index=False)
    print(f"[COMPLETADO] GAVD guardado con {len(meta_gavd)} secuencias.")

if not meta_gavd.empty and 'gait_pat' in meta_gavd.columns:
    print(meta_gavd['gait_pat'].value_counts().to_dict())

[OMITIDO] GAVD ya está descargado (41 secuencias en data\metadata\metadata_gavd.csv).
{'parkinsons': 22, 'normal': 19}


## 4. Dataset 2: UR Fall Detection (*Control Doméstico y Caídas*)

Intenta descargar las secuencias RGB directamente desde el servidor oficial de la Universidad de Rzeszów (`fenix.ur.edu.pl`). Si el servidor no responde, recurre automáticamente al repositorio en Kaggle mediante `kagglehub`.

In [6]:
def asegurar_auth_kaggle() -> None:
    """Configura las credenciales de Kaggle desde kaggle.json o entrada estándar si hace falta."""
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'

    if not kaggle_json.exists():
        for candidato in [Path('kaggle.json'), Path.home() / 'Downloads' / 'kaggle.json', Path.home() / 'Descargas' / 'kaggle.json']:
            if candidato.exists():
                kaggle_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(candidato, kaggle_json)
                break

    if kaggle_json.exists():
        creds = json.loads(kaggle_json.read_text(encoding='utf-8'))
        os.environ['KAGGLE_USERNAME'] = creds['username']
        os.environ['KAGGLE_KEY'] = creds['key']
        return

    if not os.environ.get('KAGGLE_KEY') and not os.environ.get('KAGGLE_API_TOKEN'):
        username = input('Kaggle Username: ').strip()
        key = getpass('Kaggle API Key / Token: ').strip()
        if key.startswith('KGAT_'):
            os.environ['KAGGLE_API_TOKEN'] = key
        else:
            kaggle_dir.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
            os.environ['KAGGLE_USERNAME'] = username
            os.environ['KAGGLE_KEY'] = key


def comprobar_servidor_ur() -> bool:
    """Comprueba disponibilidad del servidor HTTP oficial de UR Fall."""
    test_url = "http://fenix.ur.edu.pl/~mkepski/ds/data/adl-01-cam0-rgb.zip"
    try:
        with requests.get(test_url, headers=HTTP_HEADERS, timeout=15, verify=False, stream=True) as r:
            return r.status_code == 200
    except Exception:
        return False


def descargar_desde_ur_oficial(clips_dir: Path, max_adl: int, max_fall: int, fps: float) -> pd.DataFrame:
    """Descarga archivos ZIP oficiales de UR Fall y compila sus fotogramas en MP4."""
    seleccion = (
        [('adl', f"adl-{i:02d}", f"http://fenix.ur.edu.pl/~mkepski/ds/data/adl-{i:02d}-cam0-rgb.zip") for i in range(1, max_adl + 1)] +
        [('fall', f"fall-{i:02d}", f"http://fenix.ur.edu.pl/~mkepski/ds/data/fall-{i:02d}-cam0-rgb.zip") for i in range(1, max_fall + 1)]
    )

    temp_dir = clips_dir / '_temp_extract'
    temp_dir.mkdir(exist_ok=True)
    registros = []

    for categoria, seq_id, url in tqdm(seleccion, desc="Descargando UR Fall (oficial)"):
        mp4_path = clips_dir / f"{seq_id}.mp4"
        if not mp4_path.exists() or FORCE_DOWNLOAD:
            zip_path = temp_dir / f"{seq_id}.zip"
            extract_folder = temp_dir / seq_id
            try:
                r = requests.get(url, headers=HTTP_HEADERS, timeout=60, verify=False)
                if r.status_code != 200:
                    continue
                zip_path.write_bytes(r.content)
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(extract_folder)
                compilar_frames_a_mp4(extract_folder, mp4_path, fps=fps)
            except Exception:
                continue
            finally:
                if zip_path.exists():
                    zip_path.unlink()
                shutil.rmtree(extract_folder, ignore_errors=True)

        if mp4_path.exists() and mp4_path.stat().st_size > 1024:
            cap = cv2.VideoCapture(str(mp4_path))
            num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            registros.append({
                'seq_id': seq_id,
                'category': categoria,
                'num_frames': num_frames,
                'fps': fps,
                'clip_path': str(mp4_path)
            })

    shutil.rmtree(temp_dir, ignore_errors=True)
    return pd.DataFrame(registros)


def descargar_desde_ur_kaggle(clips_dir: Path, max_total: int, fps: float) -> pd.DataFrame:
    """Respaldo automático desde Kaggle si el servidor universitario no responde."""
    import kagglehub
    from kagglesdk import KaggleClient
    from kagglesdk.datasets.types.dataset_api_service import ApiListDatasetsRequest

    asegurar_auth_kaggle()
    client = KaggleClient()
    slug_elegido = None

    for termino in ['ur-fall', 'ur fall detection', 'up-fall']:
        req = ApiListDatasetsRequest()
        req.search = termino
        res = client.datasets.dataset_api_client.list_datasets(req)
        for ds in res.datasets:
            ref = getattr(ds, 'ref', '')
            size = getattr(ds, 'total_bytes', 0) or 0
            if size > 50 * 1024 * 1024:
                slug_elegido = ref
                break
        if slug_elegido:
            break

    if not slug_elegido:
        raise RuntimeError("No se encontró un dataset multimedia alternativo en Kaggle.")

    cache_path = Path(kagglehub.dataset_download(slug_elegido))
    registros = []

    videos = sorted([p for p in cache_path.rglob('*') if p.is_file() and p.suffix.lower() in ('.mp4', '.avi')])
    if videos:
        for idx, vid in enumerate(tqdm(videos[:max_total], desc="Copiando vídeos desde Kaggle")):
            cat = 'fall' if 'fall' in str(vid).lower() else 'adl'
            seq_id = f"{cat}-{idx+1:02d}"
            dst = clips_dir / f"{seq_id}{vid.suffix.lower()}"
            if not dst.exists() or FORCE_DOWNLOAD:
                shutil.copyfile(vid, dst)
            cap = cv2.VideoCapture(str(dst))
            num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            vid_fps = cap.get(cv2.CAP_PROP_FPS) or fps
            cap.release()
            registros.append({
                'seq_id': seq_id,
                'category': cat,
                'num_frames': num_frames,
                'fps': vid_fps,
                'clip_path': str(dst)
            })
        return pd.DataFrame(registros)

    carpetas_img = sorted({
        p.parent for p in cache_path.rglob('*')
        if p.is_file() and p.suffix.lower() in ('.png', '.jpg') and 'depth' not in str(p).lower()
    })
    for idx, folder in enumerate(tqdm(carpetas_img[:max_total], desc="Compilando secuencias desde Kaggle")):
        cat = 'fall' if 'fall' in folder.name.lower() else 'adl'
        seq_id = f"{cat}-{idx+1:02d}"
        dst = clips_dir / f"{seq_id}.mp4"
        if (dst.exists() and not FORCE_DOWNLOAD) or compilar_frames_a_mp4(folder, dst, fps=fps):
            cap = cv2.VideoCapture(str(dst))
            num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            registros.append({
                'seq_id': seq_id,
                'category': cat,
                'num_frames': num_frames,
                'fps': fps,
                'clip_path': str(dst)
            })

    return pd.DataFrame(registros)

In [7]:
if dataset_ya_descargado(META_URFALL_PATH) and not FORCE_DOWNLOAD:
    meta_urfall = pd.read_csv(META_URFALL_PATH)
    print(f"[OMITIDO] UR Fall ya está descargado ({len(meta_urfall)} secuencias en {META_URFALL_PATH}).")
else:
    print("Descargando secuencias de UR Fall...")
    if comprobar_servidor_ur():
        meta_urfall = descargar_desde_ur_oficial(
            clips_dir=URFALL_CLIPS_DIR,
            max_adl=URFALL_MAX_ADL,
            max_fall=URFALL_MAX_FALL,
            fps=URFALL_FPS
        )
    else:
        print("Servidor oficial inaccesible. Utilizando respaldo desde Kaggle...")
        meta_urfall = descargar_desde_ur_kaggle(
            clips_dir=URFALL_CLIPS_DIR,
            max_total=URFALL_MAX_ADL + URFALL_MAX_FALL,
            fps=URFALL_FPS
        )
    meta_urfall.to_csv(META_URFALL_PATH, index=False)
    print(f"[COMPLETADO] UR Fall guardado con {len(meta_urfall)} secuencias.")

if not meta_urfall.empty and 'category' in meta_urfall.columns:
    print(meta_urfall['category'].value_counts().to_dict())

[OMITIDO] UR Fall ya está descargado (35 secuencias en data\metadata\metadata_urfall.csv).
{'adl': 25, 'fall': 10}


## 5. Auditoría y Verificación Final

Comprueba la integridad de los archivos de metadatos y verifica que cada vídeo registrado se pueda abrir correctamente con OpenCV antes de pasar a la etapa de extracción de pose.

In [8]:
def auditar_dataset(nombre: str, meta_path: Path) -> None:
    if not meta_path.exists():
        print(f"[PENDIENTE] {nombre}: no existe {meta_path}")
        return

    df = pd.read_csv(meta_path)
    validos = 0
    total_frames = 0

    for ruta in df['clip_path']:
        p = Path(ruta)
        if not p.exists():
            continue
        cap = cv2.VideoCapture(str(p))
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        cap.release()
        if frames > 5 and w > 0:
            validos += 1
            total_frames += frames

    print(f"[OK] {nombre}: {validos}/{len(df)} vídeos íntegros | {total_frames} fotogramas totales.")


auditar_dataset("GAVD", META_GAVD_PATH)
auditar_dataset("UR Fall", META_URFALL_PATH)

[OK] GAVD: 41/41 vídeos íntegros | 8829 fotogramas totales.
[OK] UR Fall: 35/35 vídeos íntegros | 6774 fotogramas totales.
